# Fine-tuning con Unsloth — DPO / ORPO su Kaggle T4

**Struttura del notebook:**
1. Installazione Unsloth
2. Setup cartelle e variabili
3. Verifica GPU
4. Configurazione — **cambia `TRAINING_MODE` per passare da DPO a ORPO**
5. Caricamento dataset
6. Caricamento modello con Unsloth
7. Configurazione LoRA
8. Training
9. Salvataggio

> ⚠️ Assicurati di usare **l'acceleratore T4 GPU** in `Settings > Accelerator`.

## Cella 1 — Installazione Unsloth

In [1]:
import subprocess, sys

# Unsloth installa automaticamente la versione compatibile con la GPU presente.
# NON installare bitsandbytes separatamente: Unsloth ne porta una versione patchata.
# NON installare transformers/trl separatamente: le versioni vengono pinned da Unsloth.
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "unsloth[kaggle-new]",   # variante ottimizzata per Kaggle
    "--quiet", "--upgrade"
])

# Verifica che l'installazione sia andata a buon fine
import unsloth
print(f"✅ Unsloth {unsloth.__version__} installato")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth 2026.6.1 installato


## Cella 2 — Setup cartelle e ambiente

In [2]:
import os

# Forza Kaggle a usare solo la prima GPU (evita comportamenti strani con 2xT4)
os.environ["CUDA_VISIBLE_DEVICES"]      = "0"
os.environ["TOKENIZERS_PARALLELISM"]    = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]   = "expandable_segments:True"

BASE_DIR   = "/kaggle/working"
CACHE_DIR  = os.path.join(BASE_DIR, "hf_cache")

os.environ["HF_HOME"]            = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]  = os.path.join(CACHE_DIR, "datasets")

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"📁 Cache → {CACHE_DIR}")

📁 Cache → /kaggle/working/hf_cache


## Cella 3 — Verifica GPU

In [3]:
import torch
from unsloth import is_bfloat16_supported

print(f"CUDA disponibile    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"GPU                 : {props.name}")
    print(f"VRAM                : {vram:.1f} GB")

# T4 NON supporta BF16 — questo flag sarà usato nella config
USE_BF16 = is_bfloat16_supported()
USE_FP16 = not USE_BF16
print(f"BF16 supportato     : {USE_BF16}")
print(f"FP16 usato          : {USE_FP16}")

CUDA disponibile    : True
GPU                 : Tesla T4
VRAM                : 14.6 GB
BF16 supportato     : False
FP16 usato          : True


## Cella 4 — Configurazione

**Per passare da DPO a ORPO cambia solo `TRAINING_MODE = "orpo"`** — tutto il resto si adatta automaticamente.

In [4]:
# ─────────────────────────────────────────────────────────────
#  PARAMETRO PRINCIPALE — cambia tra "dpo" e "orpo"
# ─────────────────────────────────────────────────────────────
TRAINING_MODE = "dpo"   # oppure "orpo"

# ── Dataset ──────────────────────────────────────────────────
# Modifica con il percorso reale del tuo dataset su Kaggle
DATASET_PATH = "/kaggle/input/datasets/lorenzosalis/autobench-dpo-dataset/dpo_dataset_qwen.parquet"

# ── Modello ──────────────────────────────────────────────────
# Unsloth fornisce versioni pre-quantizzate: caricamento più veloce
# e compatibilità garantita. Usare queste invece delle HF ufficiali.
#
# Per Gemma 2 2B:
#MODEL_ID = "unsloth/gemma-2-2b-it-bnb-4bit"
#
# Per Qwen 2.5 0.5B (decommentare se vuoi Qwen):
MODEL_ID = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

# ── Iperparametri ─────────────────────────────────────────────
MAX_SEQ_LENGTH    = 512    # lunghezza massima tokenizzazione
MAX_PROMPT_LENGTH = 256    # metà di MAX_SEQ_LENGTH è un buon default

NUM_EPOCHS        = 2
BATCH_SIZE        = 2      # Unsloth permette batch > 1 grazie al minor uso di VRAM
GRAD_ACCUM        = 8      # batch effettivo = 2 * 8 = 16 samples
LEARNING_RATE     = 5e-6
BETA              = 0.1    # temperature DPO/ORPO: 0.1 è il valore standard
WARMUP_STEPS      = 0.1    # 10% degli step totali

# ── LoRA ──────────────────────────────────────────────────────
LORA_R       = 16
LORA_ALPHA   = 16   # con Unsloth: lora_alpha == r è il default raccomandato
LORA_DROPOUT = 0    # 0 è ottimizzato internamente da Unsloth

# ── Output ────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(BASE_DIR, f"model-{TRAINING_MODE}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"🎯 Modalità training : {TRAINING_MODE.upper()}")
print(f"🤖 Modello           : {MODEL_ID}")
print(f"📁 Output            : {OUTPUT_DIR}")
print(f"📏 Max seq length    : {MAX_SEQ_LENGTH}")
print(f"🔢 Batch effettivo   : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

🎯 Modalità training : DPO
🤖 Modello           : unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit
📁 Output            : /kaggle/working/model-dpo
📏 Max seq length    : 512
🔢 Batch effettivo   : 2 × 8 = 16


## Cella 5 — Autenticazione HuggingFace (richiesta per Gemma)

In [ ]:
# Gemma è un modello gated: serve token HF con accesso accettato su HuggingFace.
# Se usi Qwen puoi commentare questa cella.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Autenticato su HuggingFace")

## Cella 6 — Caricamento dataset

In [5]:
from datasets import load_dataset, load_from_disk, DatasetDict

# ── Caricamento ───────────────────────────────────────────────
# Adatta il blocco in base al formato del tuo dataset.
ext = DATASET_PATH.rsplit(".", 1)[-1].lower()
fmt_map = {"json": "json", "jsonl": "json", "csv": "csv", "parquet": "parquet"}
ds_raw = load_dataset(fmt_map.get(ext, "json"), data_files=DATASET_PATH, split="train")

# Tieni solo le colonne DPO standard
ds_raw = ds_raw.select_columns(["prompt", "chosen", "rejected"])

# ── Split train/eval ──────────────────────────────────────────
split    = ds_raw.train_test_split(test_size=0.05, seed=42)
ds_train = split["train"]
ds_eval  = split["test"]

print(f"Train : {len(ds_train):,} esempi")
print(f"Eval  : {len(ds_eval):,} esempi")
print("\nEsempio (primo record):")
print(f"  prompt   → {str(ds_train[0]['prompt'])[:80]}...")
print(f"  chosen   → {str(ds_train[0]['chosen'])[:80]}...")
print(f"  rejected → {str(ds_train[0]['rejected'])[:80]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Train : 20,774 esempi
Eval  : 1,094 esempi

Esempio (primo record):
  prompt   → [{'role': 'user', 'content': 'Provide arguments for and against the use of CRISP...
  chosen   → [{'role': 'assistant', 'content': '### **Comprehensive Analysis of CRISPR Gene E...
  rejected → [{'role': 'assistant', 'content': '**Arguments For CRISPR Gene Editing in Treati...


## Cella 7 — Caricamento modello con Unsloth

Questa è la cella che sostituisce il blocco `AutoModelForCausalLM + BitsAndBytesConfig + prepare_model_for_kbit_training` del vecchio notebook. Unsloth lo gestisce tutto internamente con kernel ottimizzati.

In [6]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,         # None = autodetect: FP16 su T4, BF16 su A100/H100
    load_in_4bit    = True,         # QLoRA 4-bit
    cache_dir       = CACHE_DIR,
    # token           = hf_token,   # Decommentare se necessario
)

# Padding: Gemma e Qwen non hanno pad_token di default
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"  # obbligatorio per DPO/ORPO

print(f"✅ Modello caricato: {MODEL_ID}")
print(f"   Pad token      : {tokenizer.pad_token!r}")
print(f"   Padding side   : {tokenizer.padding_side}")

==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
✅ Modello caricato: unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit
   Pad token      : '<|PAD_TOKEN|>'
   Padding side   : right


## Cella 8 — Configurazione LoRA con Unsloth

Nota chiave: `use_gradient_checkpointing="unsloth"` è l'opzione proprietaria di Unsloth che usa il 30% di VRAM in meno rispetto al gradient checkpointing standard di PyTorch. **Non usare `True`, usa `"unsloth"`.**

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r                     = LORA_R,
    lora_alpha            = LORA_ALPHA,
    lora_dropout          = LORA_DROPOUT,
    target_modules        = ["q_proj", "k_proj", "v_proj", "o_proj",
                              "gate_proj", "up_proj", "down_proj"],
    bias                  = "none",
    use_gradient_checkpointing = "unsloth",  # ← 30% VRAM in meno vs True
    random_state          = 42,
)

model.print_trainable_parameters()

Unsloth 2026.6.1 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## Cella 9 — Training DPO o ORPO

Il blocco `if/elif` seleziona automaticamente trainer e config in base a `TRAINING_MODE`.

**Differenze DPO vs ORPO nella config:**
- DPO: ha `precompute_ref_log_probs`, `loss_type`, `disable_dropout`
- ORPO: non ha ref model, usa `ORPOConfig` con campi simili ma senza `loss_type`
- Il dataset rimane identico per entrambi (`prompt`, `chosen`, `rejected`)

In [8]:
import torch
from unsloth import PatchDPOTrainer
from unsloth import is_bfloat16_supported

# Patching obbligatorio: applica i kernel ottimizzati di Unsloth a DPOTrainer e ORPOTrainer
PatchDPOTrainer()

torch.cuda.empty_cache()

# ── Shared config valida per entrambe le modalità ─────────────
SHARED_TRAINING_KWARGS = dict(
    output_dir                    = OUTPUT_DIR,
    num_train_epochs              = NUM_EPOCHS,
    per_device_train_batch_size   = BATCH_SIZE,
    per_device_eval_batch_size    = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LEARNING_RATE,
    lr_scheduler_type             = "cosine",
    warmup_steps                  = WARMUP_STEPS,   
    fp16                          = USE_FP16,        # True su T4, False su A100/H100
    bf16                          = USE_BF16,        # False su T4, True su A100/H100
    optim                         = "adamw_8bit",   # Unsloth raccomanda questo
    eval_strategy                 = "steps",
    eval_steps                    = 200,
    save_strategy                 = "steps",
    save_steps                    = 200,
    save_total_limit              = 2,
    load_best_model_at_end        = False,
    logging_steps                 = 10,
    report_to                     = "none",
    remove_unused_columns         = False,
    dataloader_num_workers        = 0,
    dataloader_pin_memory         = False,
    max_length                    = MAX_SEQ_LENGTH,
    max_prompt_length             = MAX_PROMPT_LENGTH,
    beta                          = BETA,
    dataset_num_proc              = 4,              # tokenizzazione parallela su CPU
)

if TRAINING_MODE == "dpo":
    from trl import DPOTrainer, DPOConfig

    config = DPOConfig(
        **SHARED_TRAINING_KWARGS,
        truncation_mode           = "keep_start",
        loss_type                 = "sigmoid",
        label_smoothing           = 0.0,
        disable_dropout           = True,
        precompute_ref_log_probs  = False,  # ← elimina il ref model dalla VRAM durante il train
        #precompute_ref_batch_size = 4,
    )

    trainer = DPOTrainer(
        model            = model,
        ref_model        = None,   # con LoRA+precompute non serve un modello separato
        args             = config,
        train_dataset    = ds_train,
        eval_dataset     = ds_eval,
        processing_class = tokenizer,
    )

elif TRAINING_MODE == "orpo":
    from trl import ORPOTrainer, ORPOConfig

    # ORPOConfig non ha loss_type, disable_dropout, precompute_ref_log_probs
    # (ORPO non usa un ref model per definizione)
    orpo_kwargs = {k: v for k, v in SHARED_TRAINING_KWARGS.items()
                   if k not in ["max_length", "max_prompt_length", "beta",
                                 "dataset_num_proc"]}

    config = ORPOConfig(
        **orpo_kwargs,
        max_length             = MAX_SEQ_LENGTH,
        max_prompt_length      = MAX_PROMPT_LENGTH,
        beta                   = BETA,
        dataset_num_proc       = 4,
    )

    trainer = ORPOTrainer(
        model            = model,
        args             = config,
        train_dataset    = ds_train,
        eval_dataset     = ds_eval,
        processing_class = tokenizer,
    )

else:
    raise ValueError(f"TRAINING_MODE deve essere 'dpo' o 'orpo', ricevuto: '{TRAINING_MODE}'")

print(f"🚀 Avvio training {TRAINING_MODE.upper()} con Unsloth...")
print(f"   Batch effettivo   : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Epoche            : {NUM_EPOCHS}")
print(f"   FP16/BF16         : fp16={USE_FP16}, bf16={USE_BF16}")

train_result = trainer.train()

print("\n📊 Risultati training:")
print(train_result)

Extracting prompt in train dataset (num_proc=4):   0%|          | 0/20774 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=4):   0%|          | 0/20774 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=4):   0%|          | 0/20774 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=4):   0%|          | 0/1094 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=4):   0%|          | 0/1094 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,774 | Num Epochs = 2 | Total steps = 2,598
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
200,0.656500,0.638609,0.299579,0.160949,0.682815,0.138630,-681.504089,-618.699951,-1.173969,-1.247655
400,0.573802,0.561747,0.487256,0.126803,0.778793,0.360453,-679.627258,-619.041504,-1.159226,-1.229796
600,0.499314,0.500971,0.622768,0.041765,0.811700,0.581002,-678.272095,-619.891907,-1.178341,-1.247876
800,0.472464,0.455078,0.710351,-0.027260,0.845521,0.737611,-677.396362,-620.582092,-1.179422,-1.248960
1000,0.426265,0.416433,0.603159,-0.261920,0.867459,0.865080,-678.468262,-622.928772,-1.250963,-1.322775
1200,0.414529,0.383887,0.585634,-0.414065,0.891225,0.999700,-678.643433,-624.450134,-1.293782,-1.366462
1400,0.355826,0.356605,0.528571,-0.585842,0.904936,1.114413,-679.214172,-626.167908,-1.293520,-1.366776
1600,0.347580,0.338267,0.548087,-0.658612,0.913163,1.206699,-679.018982,-626.895630,-1.306963,-1.380100
1800,0.301518,0.325248,0.571947,-0.708085,0.912249,1.280032,-678.780457,-627.390320,-1.308878,-1.381717
2000,0.267348,0.317711,0.539805,-0.782843,0.917733,1.322648,-679.101868,-628.137939,-1.326793,-1.400183


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


📊 Risultati training:
TrainOutput(global_step=2598, training_loss=0.414196835379127, metrics={'train_runtime': 20619.6978, 'train_samples_per_second': 2.015, 'train_steps_per_second': 0.126, 'total_flos': 0.0, 'train_loss': 0.414196835379127, 'epoch': 2.0})


## Cella 10 — Salvataggio adapter LoRA

In [9]:
FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

# Salva solo gli adapter LoRA (pochi MB), non i pesi base del modello
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Salva metriche e stato
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"✅ Adapter LoRA salvato in: {FINAL_DIR}")
print("\nFile salvati:")
for f in sorted(os.listdir(FINAL_DIR)):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024**2
    print(f"  {f:45s} {size:.1f} MB")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/model-dpo/final/tokenizer_config.json.


***** train metrics *****
  epoch                    =        2.0
  total_flos               =        0GF
  train_loss               =     0.4142
  train_runtime            = 5:43:39.69
  train_samples_per_second =      2.015
  train_steps_per_second   =      0.126
✅ Adapter LoRA salvato in: /kaggle/working/model-dpo/final

File salvati:
  README.md                                     0.0 MB
  adapter_config.json                           0.0 MB
  adapter_model.safetensors                     33.6 MB
  chat_template.jinja                           0.0 MB
  tokenizer.json                                10.9 MB
  tokenizer_config.json                         0.0 MB
